In [1]:
import json

with open("./full_format_recipes.json", "r") as f:
    recipes = json.load(f)

In [2]:
recipes = recipes [:1000]

In [3]:
recipes [0]

{'directions': ['1. Place the stock, lentils, celery, carrot, thyme, and salt in a medium saucepan and bring to a boil. Reduce heat to low and simmer until the lentils are tender, about 30 minutes, depending on the lentils. (If they begin to dry out, add water as needed.) Remove and discard the thyme. Drain and transfer the mixture to a bowl; let cool.',
  '2. Fold in the tomato, apple, lemon juice, and olive oil. Season with the pepper.',
  '3. To assemble a wrap, place 1 lavash sheet on a clean work surface. Spread some of the lentil mixture on the end nearest you, leaving a 1-inch border. Top with several slices of turkey, then some of the lettuce. Roll up the lavash, slice crosswise, and serve. If using tortillas, spread the lentils in the center, top with the turkey and lettuce, and fold up the bottom, left side, and right side before rolling away from you.'],
 'fat': 7.0,
 'date': '2006-09-01T04:00:00.000Z',
 'categories': ['Sandwich',
  'Bean',
  'Fruit',
  'Tomato',
  'turkey',

In [4]:
recipes [0] ["ingredients"]

['4 cups low-sodium vegetable or chicken stock',
 '1 cup dried brown lentils',
 '1/2 cup dried French green lentils',
 '2 stalks celery, chopped',
 '1 large carrot, peeled and chopped',
 '1 sprig fresh thyme',
 '1 teaspoon kosher salt',
 '1 medium tomato, cored, seeded, and diced',
 '1 small Fuji apple, cored and diced',
 '1 tablespoon freshly squeezed lemon juice',
 '2 teaspoons extra-virgin olive oil',
 'Freshly ground black pepper to taste',
 '3 sheets whole-wheat lavash, cut in half crosswise, or 6 (12-inch) flour tortillas',
 '3/4 pound turkey breast, thinly sliced',
 '1/2 head Bibb lettuce']

In [5]:
from ingredient_parser import parse_ingredient

for ingredient in recipes [0] ["ingredients"]:
    parsed_ingredient = parse_ingredient(ingredient)

    print(parsed_ingredient.name)
    print(parsed_ingredient.amount)

[IngredientText(text='low-sodium vegetable stock', confidence=0.972664, starting_index=2), IngredientText(text='low-sodium chicken stock', confidence=0.950144, starting_index=2)]
[IngredientAmount(quantity=Fraction(4, 1), quantity_max=Fraction(4, 1), unit=<Unit('cup')>, text='4 cups', confidence=0.999851, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)]
[IngredientText(text='dried brown lentils', confidence=0.987636, starting_index=2)]
[IngredientAmount(quantity=Fraction(1, 1), quantity_max=Fraction(1, 1), unit=<Unit('cup')>, text='1 cup', confidence=0.999969, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)]
[IngredientText(text='dried French green lentils', confidence=0.996553, starting_index=2)]
[IngredientAmount(quantity=Fraction(1, 2), quantity_max=F

In [6]:
import sys
sys.path.append("..")

from food_data_extraction import FoodDensityEmbedding

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


In [7]:
food_density_embedding = FoodDensityEmbedding()

In [8]:
food_density_embedding.load(index_path = "../food_data_extraction/food_density_embedding.faiss")

In [9]:
def calculate_volume(unit: str, quantity: float) -> float | None:
    if unit == "cup":
        return quantity * 236.588
    elif unit == "tablespoon":
        return quantity * 14.7868
    elif unit == "teaspoon":
        return quantity * 4.92892
    
    return None

In [10]:
# density = mass / volume
# mass = density * voume

def calculate_weight(unit: str, quantity: float, food_name: str) -> float | None:
    search_results = food_density_embedding.search(food_name)

    if search_results.empty:
        density = 1.0 # TODO see this
    else:
        density = search_results ["density (g/cm^3)"].iloc [0]

    volume = calculate_volume(unit, quantity)

    if volume is None:
        return None

    return density * volume

In [11]:
print(calculate_weight("cup", 1, "cooked brown rice"), "g")

170.0 g


In [28]:
test = parse_ingredient("1 1/2 cups brown rice")

In [29]:
print(test.amount [0])

IngredientAmount(quantity=Fraction(3, 2), quantity_max=Fraction(3, 2), unit=<Unit('cup')>, text='1 1/2 cups', confidence=0.999983, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)


In [ ]:
from ingredient_parser.dataclasses import UREG

density = UREG.Quantity(0.911, "g/cm^3")  # butter density
print(test.amount[0].convert_to("g", densisty=density))

IngredientAmount(quantity=323.29782517724993, quantity_max=323.29782517724993, unit=<Unit('gram')>, text='323.298 gram', confidence=0.999983, starting_index=0, unit_system=<UnitSystem.METRIC: 'metric'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)


In [ ]:
from tqdm import tqdm

parsed_recipes = []

for recipe in tqdm(recipes, desc = "Parsing recipes"):
    parsed_recipe = recipe.copy()

    try:
        parsed_ingredients = [parse_ingredient(ingredient) for ingredient in recipe ["ingredients"]]
    except ValueError:
        continue

    try:
        ingredient_names = [ingredient.name [0].text for ingredient in parsed_ingredients] # TODO for now only taking the first name, so not considering if it has "Or"

        parsed_recipe ["ingredient_names"] = ingredient_names

        parsed_ingredient_quantities = [ingredient.amount [0] if ingredient.amount else None for ingredient in parsed_ingredients] # TODO for now only taking the first quantity, so not considering if it has "Or", but this might not be as problematic as the name, since the quantity is usually the same for both options

        resolved_ingredient_quantities = []
        ingredient_weights = []

        for ingredient_name, parsed_quantity in zip(ingredient_names, parsed_ingredient_quantities):
            if parsed_quantity is None:
                ingredient_weights.append(None) # TODO later on, use the portion.csv to resolve these, by taking the average quantity for that ingredient and unit
                resolved_ingredient_quantities.append(None)
                continue

            print(parsed_quantity)

            ingredient_weights.append(calculate_weight(parsed_quantity.unit, parsed_quantity.quantity, ingredient_name))

            if not hasattr(parsed_quantity, "quantity") or not hasattr(parsed_quantity, "quantity_max"):
                resolved_ingredient_quantities.append(None) # TODO later on, use the portion.csv to resolve these, by taking the average quantity for that ingredient and unit
                continue

            if parsed_quantity.quantity == "" and parsed_quantity.quantity_max == "":
                resolved_ingredient_quantities.append(1)
                continue

            resolved_ingredient_quantities.append((float(parsed_quantity.quantity) + float(parsed_quantity.quantity_max)) / 2)

        parsed_recipe ["ingredient_quantities"] = resolved_ingredient_quantities
        parsed_recipe ["ingredient_weights"] = ingredient_weights

        parsed_recipes.append(parsed_recipe)

    except IndexError:
        print(f"IndexError parsing ingredients for recipe {recipe ['title']}") 
        continue # TODO investigate

Parsing recipes:   0%|          | 2/1000 [00:00<01:39, 10.04it/s]

IngredientAmount(quantity=Fraction(4, 1), quantity_max=Fraction(4, 1), unit=<Unit('cup')>, text='4 cups', confidence=0.999851, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)
IngredientAmount(quantity=Fraction(1, 1), quantity_max=Fraction(1, 1), unit=<Unit('cup')>, text='1 cup', confidence=0.999969, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)
IngredientAmount(quantity=Fraction(1, 2), quantity_max=Fraction(1, 2), unit=<Unit('cup')>, text='1/2 cups', confidence=0.999962, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)
IngredientAmount(quantity=Fraction(2, 1), quantity_max=Fraction(2, 1), unit='stalks', text='2 stalks', confidence=0.9999

Parsing recipes:   0%|          | 5/1000 [00:00<01:04, 15.33it/s]

IngredientAmount(quantity=Fraction(1, 1), quantity_max=Fraction(1, 1), unit=<Unit('cup')>, text='1 cup', confidence=0.999864, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=True)
IngredientAmount(quantity=Fraction(1, 1), quantity_max=Fraction(1, 1), unit=<Unit('cup')>, text='1 cup', confidence=0.999958, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)
IngredientAmount(quantity=Fraction(1, 1), quantity_max=Fraction(1, 1), unit=<Unit('teaspoon')>, text='1 teaspoon', confidence=0.99994, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)
IngredientAmount(quantity=Fraction(2, 1), quantity_max=Fraction(2, 1), unit='cans', text='2 cans', confidence=0.9997

Parsing recipes:   1%|          | 8/1000 [00:00<00:52, 18.86it/s]

...

Parsing recipes:   1%|▏         | 13/1000 [00:00<00:54, 18.06it/s]

IndexError parsing ingredients for recipe Spicy Noodle Soup 
IngredientAmount(quantity=Fraction(3, 1), quantity_max=Fraction(3, 1), unit=<Unit('cup')>, text='3 cups', confidence=0.999967, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)
IngredientAmount(quantity=Fraction(2, 1), quantity_max=Fraction(2, 1), unit=<Unit('teaspoon')>, text='2 teaspoons', confidence=0.999976, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)
IngredientAmount(quantity=Fraction(2, 1), quantity_max=Fraction(2, 1), unit=<Unit('teaspoon')>, text='2 teaspoons', confidence=0.999982, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)
IngredientAmount(quantity=Fraction(3, 2)

Parsing recipes:   2%|▏         | 16/1000 [00:00<00:57, 17.10it/s]


IngredientAmount(quantity=Fraction(1, 4), quantity_max=Fraction(1, 4), unit=<Unit('cup')>, text='1/4 cups', confidence=0.999919, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)
IngredientAmount(quantity=Fraction(1, 4), quantity_max=Fraction(1, 4), unit=<Unit('cup')>, text='1/4 cups', confidence=0.999943, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=True)
IngredientAmount(quantity=Fraction(2, 1), quantity_max=Fraction(2, 1), unit=<Unit('tablespoon')>, text='2 tablespoons', confidence=0.999881, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=True)
IngredientAmount(quantity=Fraction(2, 1), quantity_max=Fraction(2, 1), unit=<Unit('tablespoon')>, text='2

AttributeError: 'CompositeIngredientAmount' object has no attribute 'unit'

In [40]:

import json
# print(parsed_recipes [0])

print(json.dumps(parsed_recipes [0], indent = 4))

{
    "directions": [
        "1. Place the stock, lentils, celery, carrot, thyme, and salt in a medium saucepan and bring to a boil. Reduce heat to low and simmer until the lentils are tender, about 30 minutes, depending on the lentils. (If they begin to dry out, add water as needed.) Remove and discard the thyme. Drain and transfer the mixture to a bowl; let cool.",
        "2. Fold in the tomato, apple, lemon juice, and olive oil. Season with the pepper.",
        "3. To assemble a wrap, place 1 lavash sheet on a clean work surface. Spread some of the lentil mixture on the end nearest you, leaving a 1-inch border. Top with several slices of turkey, then some of the lettuce. Roll up the lavash, slice crosswise, and serve. If using tortillas, spread the lentils in the center, top with the turkey and lettuce, and fold up the bottom, left side, and right side before rolling away from you."
    ],
    "fat": 7.0,
    "date": "2006-09-01T04:00:00.000Z",
    "categories": [
        "Sandwi

In [32]:
for ingredient_name, ingredient_quantity in zip(parsed_recipes [0] ["ingredient_names"], parsed_recipes [0] ["ingredient_quantities"]):
    print(ingredient_name)
    print(ingredient_quantity)
    print("")

low-sodium vegetable stock
IngredientAmount(quantity=Fraction(4, 1), quantity_max=Fraction(4, 1), unit=<Unit('cup')>, text='4 cups', confidence=0.999851, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)

dried brown lentils
IngredientAmount(quantity=Fraction(1, 1), quantity_max=Fraction(1, 1), unit=<Unit('cup')>, text='1 cup', confidence=0.999969, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)

dried French green lentils
IngredientAmount(quantity=Fraction(1, 2), quantity_max=Fraction(1, 2), unit=<Unit('cup')>, text='1/2 cups', confidence=0.999962, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)

celery
IngredientAmount(quantity=Fraction(2

In [23]:
# ingredient_weights = []

# for quantity in parsed_recipes [0] ["ingredient_quantities"]:
#     print(quantity)

#     if quantity is None:
#         # TODO take the food_portion.csv file and use the serving size

#         continue

#     # take average between quantity and quantity max
#     if quantity.quantity_max is not None:
#         quantity_value = float((quantity.quantity + quantity.quantity_max) / 2)
#     else:
#         quantity_value = float(quantity.quantity)

IngredientAmount(quantity=Fraction(4, 1), quantity_max=Fraction(4, 1), unit=<Unit('cup')>, text='4 cups', confidence=0.999851, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)
Resolved quantity value: 4.0
cup
IngredientAmount(quantity=Fraction(1, 1), quantity_max=Fraction(1, 1), unit=<Unit('cup')>, text='1 cup', confidence=0.999969, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)
Resolved quantity value: 1.0
cup
IngredientAmount(quantity=Fraction(1, 2), quantity_max=Fraction(1, 2), unit=<Unit('cup')>, text='1/2 cups', confidence=0.999962, starting_index=0, unit_system=<UnitSystem.US_CUSTOMARY: 'us_customary'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)
Resolved quantity value: 0.5
cup
IngredientAmount(quan

In [ ]:
from food_data_extraction import FoodEmbedding

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


In [12]:
food_embedding = FoodEmbedding()

c:\Users\micha\Documents\School\University\3rd Year\Final Year Project\Code\epicurious_dataset\..\food_data_extraction\FoodEmbedding.py:21: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  self.food_nutrient = pd.read_csv(f"{os.path.join(base_dir, 'FoodData_Central_csv_2025-12-18')}/food_nutrient.csv")


In [ ]:
food_embedding.load(index_path = "../food_data_extraction/food_embedding.faiss")

In [14]:
# display(search_fdc("Classic mixed vegetables, canned, reduced sodium, cooked with oil"))

In [21]:
nutrient_to_id = {
    "calories": 1008,
    "carbohydrates": 1005,
    "sugar": 1063,
    "protein": 1003,
    "fat": 1004,
    "saturated_fat": 1258,
    "fiber": 1079,
    "sodium": 1093
}

In [ ]:
from models import Ingredient, Recipe, NutritionalInformation

for ingredient in parsed_recipes [0] ["ingredients"]:
    print("\nActual ingredient:", ingredient)

    fdc_food = food_embedding.search(ingredient)

    print(f"Retrieved ingredient: {fdc_food.iloc[0].description} (ID: {fdc_food.iloc[0].fdc_id})")

    fdc_food_result = food_embedding.search(ingredient).iloc [0]

    fdc_nutritional_information = food_embedding.get_nutritional_information(fdc_food_result.fdc_id)

    # print(fdc_nutritional_information)

    # calories = fdc_nutritional_information [fdc_nutritional_information ["nutrient_id"] == 1008]
    # carbohydrates = fdc_nutritional_information [fdc_nutritional_information ["nutrient_id"] == 1005]
    # sugar = fdc_nutritional_information [fdc_nutritional_information ["nutrient_id"] == 1063]
    # protein = fdc_nutritional_information [fdc_nutritional_information ["nutrient_id"] == 1003]
    # fat = fdc_nutritional_information [fdc_nutritional_information ["nutrient_id"] == 1004]
    # saturated_fat = fdc_nutritional_information [fdc_nutritional_information ["nutrient_id"] == 1258]
    # fiber = fdc_nutritional_information [fdc_nutritional_information ["nutrient_id"] == 1079]
    # sodium = fdc_nutritional_information [fdc_nutritional_information ["nutrient_id"] == 1093]

    nutrient_values = {}

    for nutrient_name, nutrient_id in nutrient_to_id.items():
        nutrient = fdc_nutritional_information [fdc_nutritional_information ["nutrient_id"] == nutrient_id]

        if not nutrient.empty:
            nutrient_values [nutrient_name] = nutrient.iloc [0] ["amount"]
        else:
            nutrient_values [nutrient_name] = None

    # TODO for now all ingredients are not beign checked for gluten free, vegan, etc

    # should fallback by checking for another id, like sugar has 2000 as well

    print(nutrient_values)

    # Now calculate how much of each nutrient is in the ingredient based on the amount in the recipe, and the serving size of the food in FDC

    print(parsed_recipes [0])

    # print(f"Number of calories for {ingredient} is {calories [0]} kcal")


Actual ingredient: low-sodium vegetable stock
Retrieved ingredient: ORGANIC LOW SODIUM VEGETABLE STOCK (ID: 1111545)
{'calories': np.float64(2.0), 'carbohydrates': np.float64(0.42), 'sugar': None, 'protein': np.float64(0.0), 'fat': np.float64(0.0), 'saturated_fat': np.float64(0.0), 'fiber': np.float64(0.0), 'sodium': np.float64(15.0)}
{'directions': ['1. Place the stock, lentils, celery, carrot, thyme, and salt in a medium saucepan and bring to a boil. Reduce heat to low and simmer until the lentils are tender, about 30 minutes, depending on the lentils. (If they begin to dry out, add water as needed.) Remove and discard the thyme. Drain and transfer the mixture to a bowl; let cool.', '2. Fold in the tomato, apple, lemon juice, and olive oil. Season with the pepper.', '3. To assemble a wrap, place 1 lavash sheet on a clean work surface. Spread some of the lentil mixture on the end nearest you, leaving a 1-inch border. Top with several slices of turkey, then some of the lettuce. Roll u

In [ ]:
print("test")